# Part 2 — Advanced RAG: Hybrid Search + Cross-Encoder Reranking

**Series:** Agentic RAG with LangGraph — ArXiv ML/AI Research Paper Q&A  
**Notebook:** 2 of 3  
**Prerequisite:** Run `01_naive_rag.ipynb` first — this notebook loads the FAISS index built there.

---

## What you will build

By the end of this notebook you will have:

1. Understood why dense-only retrieval fails for keyword and exact-match queries
2. Built a **BM25 retriever** from scratch — the classic keyword-based search algorithm
3. Combined dense + BM25 into a **hybrid retriever** with alpha-weighted score fusion
4. Added a **cross-encoder reranker** (ms-marco-MiniLM) that re-scores the candidates with full query-document attention
5. Ablated chunk size and measured the impact on retrieval quality
6. Compared all three retrieval strategies in a **results table**

---

## Prerequisites — what you need to know before starting

### Knowledge prerequisites

| Concept | Where it was introduced |
|---------|------------------------|
| Embeddings, cosine similarity | Notebook 01, Lesson 4 |
| FAISS index, dense retrieval | Notebook 01, Lessons 3–5 |
| RAG pipeline overview | Notebook 01, Lesson 2 |

**New concepts introduced in this notebook:**
- BM25 scoring (term frequency × inverse document frequency)
- Hybrid retrieval and score fusion
- Cross-encoder vs. bi-encoder architecture
- Two-stage retrieval (retrieve → rerank)

### Tool prerequisites

All tools from notebook 01 plus:
```bash
# rank_bm25 and sentence-transformers are already in requirements.txt
# The cross-encoder model downloads automatically on first use (~22 MB)
```

---

## Lesson 1 — The two failure modes of dense-only retrieval

### Failure mode 1: exact-term queries

Dense embeddings encode *semantic meaning*. This is powerful for paraphrase queries but creates a blind spot for exact terms:

- Query: `"BM25 algorithm"` → the embedding represents "scoring function for text search"
- The embedding space might place "BM25" near other retrieval algorithms, but it loses the *exact string*
- A paper that only mentions "BM25" once might rank lower than a paper about retrieval in general

**Keywords, model names, paper titles, and acronyms are often better served by exact-match (sparse) retrieval.**

### Failure mode 2: retrieval quality cap

Dense bi-encoders produce embeddings independently for the query and each document. They cannot model direct interactions between query words and document words — each is compressed into its own vector before any comparison happens.

```
Bi-encoder (used for retrieval — FAST but approximate):
    query ──► encoder ──► q_vec ─┐
                                  ├── dot product ──► score
    doc   ──► encoder ──► d_vec ─┘
    (query and document never "see" each other during encoding)

Cross-encoder (used for reranking — SLOW but accurate):
    [query] [SEP] [document] ──► encoder ──► relevance score
    (full bidirectional attention between all query and document tokens)
```

**Strategy:** use the fast bi-encoder to retrieve a candidate set (e.g. top-50), then use the accurate cross-encoder to re-rank those 50 candidates to precision (top-5).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
INDEX_DIR = ARTIFACTS_DIR / "faiss_index"
EVAL_DIR = ARTIFACTS_DIR / "eval_results"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loguru import logger

from src.ingest import (
    load_index_and_chunks, load_arxiv_papers, chunk_documents,
    EMBED_MODEL_PRIMARY, EMBED_MODEL_LITE,
)
from src.retriever import DenseRetriever, BM25Retriever, HybridRetriever, Reranker
from src.evaluator import compute_retrieval_metrics, EvalResults

# Load the index built in notebook 01
faiss_index, chunks = load_index_and_chunks(INDEX_DIR)

# Must match the embed_model used when building the index in notebook 01
EMBED_MODEL = EMBED_MODEL_LITE   # qwen3-embedding:0.6b — matches the saved index

# Rebuild the paper list for eval set construction (fast — just metadata)
papers = load_arxiv_papers(n_samples=300)

print(f"Loaded index: {faiss_index.ntotal} vectors, dim={faiss_index.d}")
print(f"Loaded chunks: {len(chunks)}")
print(f"Embed model  : {EMBED_MODEL}")

In [ ]:
# Rebuild the eval set — same queries as notebook 01 (adapted to our corpus)
def find_relevant_ids_by_keyword(keyword, papers, top_n=3):
    kw = keyword.lower()
    return [p["id"] for p in papers if kw in p["title"].lower() or kw in p["abstract"].lower()][:top_n]

eval_queries = [
    {"question": "What is attention head heterogeneity in transformers?",
     "relevant_ids": find_relevant_ids_by_keyword("attention head", papers)},
    {"question": "How does contrastive learning work?",
     "relevant_ids": find_relevant_ids_by_keyword("contrastive learning", papers)},
    {"question": "What is dynamic batching for LLM inference?",
     "relevant_ids": find_relevant_ids_by_keyword("dynamic batching", papers)},
    {"question": "How do diffusion models generate images?",
     "relevant_ids": find_relevant_ids_by_keyword("diffusion model", papers)},
    {"question": "What are calibration methods for neural networks?",
     "relevant_ids": find_relevant_ids_by_keyword("calibration", papers)},
]

for q in eval_queries:
    found = sum(1 for p in papers if p["id"] in q["relevant_ids"])
    print(f"Q: {q['question'][:55]}  →  {found} relevant papers")

# Load baseline from notebook 01 for comparison
try:
    baseline = EvalResults.load(EVAL_DIR / "01_naive_rag.json")
    print("\nBaseline (naive RAG dense-only):")
    for k, v in baseline.retrieval_metrics.items():
        print(f"  {k}: {v}")
except FileNotFoundError:
    print("\nBaseline not found — run notebook 01 first")
    baseline = None

---

## Lesson 2 — BM25: how keyword search actually works

**BM25 (Best Match 25)** is a probabilistic retrieval model. Despite being from 1994, it remains competitive with dense retrieval on many benchmarks — especially for keyword-heavy queries.

### The BM25 formula

For a query Q with terms t₁, t₂, ..., tₙ, and a document D:

```
BM25(D, Q) = Σ IDF(tᵢ) × [tf(tᵢ, D) × (k₁ + 1)] / [tf(tᵢ, D) + k₁ × (1 - b + b × |D|/avgdl)]
```

Where:
- **IDF(t)** = log((N - df(t) + 0.5) / (df(t) + 0.5))
  - N = total documents; df(t) = documents containing term t
  - Rare terms (low df) get high IDF → they're more discriminative
  - Common terms ("the", "is") get near-zero IDF → essentially ignored
- **tf(t, D)** = how many times term t appears in document D
  - But saturated: doubling tf doesn't double the score (unlike raw TF-IDF)
  - k₁ = 1.5 controls saturation
- **Length normalisation**: `b × |D|/avgdl`
  - A long document mentioning "transformer" 10 times is less impressive than a short document mentioning it 3 times
  - b = 0.75 controls how much length penalises the score

### In plain English

> A document scores highly if it contains the query terms frequently (but with diminishing returns), those terms are rare in the corpus (so they're meaningful), and the document isn't unusually long.

In [ ]:
# Build the BM25 index over all chunks
# rank_bm25 tokenises by whitespace — simple but effective for abstracts

bm25_retriever = BM25Retriever(chunks=chunks)

# Test with a query that contains specific terminology
keyword_query = "BM25 sparse retrieval TF-IDF"
bm25_results = bm25_retriever.retrieve(keyword_query, k=5)

print(f"BM25 results for: '{keyword_query}'\n")
for i, r in enumerate(bm25_results, 1):
    print(f"[{i}] Score: {r['score']:.4f}")
    print(f"    Title: {r['title'][:70]}")
    print(f"    Text : {r['text'][:180]}...")
    print()

In [ ]:
# Evaluate BM25 alone and compare to dense baseline
dense_retriever = DenseRetriever(faiss_index, chunks, embed_model=EMBED_MODEL)
bm25_metrics = compute_retrieval_metrics(eval_queries, bm25_retriever, k=5)

print("BM25 only — Retrieval Metrics")
print("=" * 40)
for k, v in bm25_metrics.items():
    print(f"  {k:20s}: {v}")

if baseline is not None:
    print("\nComparison vs. dense baseline:")
    for key in ["recall@5", "precision@5", "mrr"]:
        base_v = baseline.retrieval_metrics.get(key, "N/A")
        new_v  = bm25_metrics.get(key, 0)
        delta  = new_v - base_v if isinstance(base_v, float) else 0
        direction = "+" if delta >= 0 else ""
        print(f"  {key:20s}: {base_v:.4f} → {new_v:.4f}  ({direction}{delta:.4f})")

---

## Lesson 3 — Hybrid search: combining dense and sparse

Neither BM25 nor dense retrieval dominates on all query types:

| Query type | Dense wins | BM25 wins |
|-----------|------------|----------|
| "What is the mechanism behind attention?" | ✓ (semantic) | ✗ |
| "papers mentioning BM25 Okapi" | ✗ | ✓ (exact term) |
| "Qwen3 model architecture" | ✗ | ✓ (proper noun) |
| "how do LLMs reason step by step" | ✓ (chain-of-thought paraphrase) | ✗ |

**Hybrid search** combines both score signals:

```
hybrid_score = α × dense_score_normalised + (1-α) × bm25_score_normalised
```

With α=0.7: 70% weight on semantic similarity, 30% on keyword overlap. Both scores are min-max normalised to [0,1] before blending so they're on the same scale.

We also implement **Reciprocal Rank Fusion (RRF)** as an alternative — it ignores raw scores entirely and only uses rank positions, making it robust to score scale differences between retrievers.

In [ ]:
# Build hybrid retriever (alpha=0.7: 70% dense, 30% BM25)
hybrid_retriever = HybridRetriever(
    dense=dense_retriever,
    bm25=bm25_retriever,
    alpha=0.7,
    fusion="alpha",
)

# Run on the eval set
hybrid_metrics = compute_retrieval_metrics(eval_queries, hybrid_retriever, k=5)

print("Hybrid retrieval (alpha=0.7) — Metrics")
print("=" * 40)
for k, v in hybrid_metrics.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# Ablate alpha: sweep from 0.0 (BM25 only) to 1.0 (dense only)
alphas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_recalls = []

for alpha in alphas:
    ret = HybridRetriever(dense_retriever, bm25_retriever, alpha=alpha)
    m = compute_retrieval_metrics(eval_queries, ret, k=5)
    alpha_recalls.append(m["recall@5"])

# Plot
plt.figure(figsize=(8, 4))
plt.plot(alphas, alpha_recalls, marker="o", color="steelblue", linewidth=2)
plt.axvline(alphas[alpha_recalls.index(max(alpha_recalls))], color="red", linestyle="--",
            label=f"Best alpha={alphas[alpha_recalls.index(max(alpha_recalls))]}")
plt.xlabel("Alpha (0=BM25 only, 1=Dense only)")
plt.ylabel("Recall@5")
plt.title("Hybrid Retrieval: Alpha Ablation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(EVAL_DIR / "alpha_ablation.png", dpi=150)
plt.show()

best_alpha = alphas[alpha_recalls.index(max(alpha_recalls))]
print(f"Best alpha: {best_alpha}  →  Recall@5: {max(alpha_recalls):.4f}")

---

## Lesson 4 — Cross-encoder reranking: two-stage retrieval

### Why reranking works

After hybrid retrieval you have, say, the top-20 candidates. Their ranking is imperfect because bi-encoders and BM25 both have limitations. A cross-encoder can *re-evaluate* each candidate with full query-document interaction:

```
Stage 1 (fast): Retrieve 20 candidates from hybrid retriever
                Time: ~5ms (FAISS + BM25)

Stage 2 (accurate): Cross-encoder scores all 20 (query, doc) pairs
                    Time: ~200ms on CPU for 20 pairs (worth it!)
                    Returns top-5 re-sorted by true relevance
```

### Model: cross-encoder/ms-marco-MiniLM-L-6-v2

- Trained on **MS MARCO** — 530K+ query-passage pairs from real Bing search queries
- Architecture: BERT-style with `[CLS] query [SEP] document [SEP]` input
- 22 MB on disk, 6 transformer layers, runs comfortably on CPU
- Output: a single logit (higher = more relevant)
- Strong zero-shot generalisation to new domains despite being trained on web search

In [ ]:
# Initialise the cross-encoder reranker
# First call downloads the model weights (~22 MB)
reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Demo: fetch 20 candidates with hybrid, rerank to top-5
test_query = "How does RLHF train language models using human feedback?"

# Step 1: retrieve 20 candidates
candidates = hybrid_retriever.retrieve(test_query, k=20)
print(f"Before reranking (top 3 of 20):")
for i, c in enumerate(candidates[:3], 1):
    print(f"  [{i}] score={c['score']:.4f} | {c['title'][:60]}")

print()

# Step 2: rerank to top-5
reranked = reranker.rerank(test_query, candidates, top_k=5)
print(f"After reranking (top 3 of 5):")
for i, c in enumerate(reranked[:3], 1):
    print(f"  [{i}] score={c['score']:.4f} | {c['title'][:60]}")

In [ ]:
# Evaluate hybrid + reranking pipeline
# We need a custom evaluate loop because the reranker wraps the retriever

from src.evaluator import recall_at_k, precision_at_k, mean_reciprocal_rank

recalls, precisions, mrrs = [], [], []

for q in eval_queries:
    # Stage 1: hybrid retrieves 20
    candidates = hybrid_retriever.retrieve(q["question"], k=20)
    # Stage 2: reranker selects top-5
    final = reranker.rerank(q["question"], candidates, top_k=5)
    retrieved_ids = [r["paper_id"] for r in final]

    recalls.append(recall_at_k(retrieved_ids, q["relevant_ids"], k=5))
    precisions.append(precision_at_k(retrieved_ids, q["relevant_ids"], k=5))
    mrrs.append(mean_reciprocal_rank(retrieved_ids, q["relevant_ids"]))

reranked_metrics = {
    "recall@5": round(sum(recalls) / len(recalls), 4),
    "precision@5": round(sum(precisions) / len(precisions), 4),
    "mrr": round(sum(mrrs) / len(mrrs), 4),
}

print("Hybrid + Reranking — Metrics")
print("=" * 40)
for k, v in reranked_metrics.items():
    print(f"  {k:20s}: {v}")

---

## Final comparison: all three retrieval strategies

In [ ]:
# Build the comparison table
dense_metrics = compute_retrieval_metrics(eval_queries, dense_retriever, k=5)

comparison = pd.DataFrame({
    "Strategy": ["Dense only (baseline)", "BM25 only", "Hybrid (α=0.7)", "Hybrid + Rerank"],
    "Recall@5":    [baseline.retrieval_metrics.get("recall@5", 0), bm25_metrics["recall@5"],
                    hybrid_metrics["recall@5"], reranked_metrics["recall@5"]],
    "Precision@5": [baseline.retrieval_metrics.get("precision@5", 0), bm25_metrics["precision@5"],
                    hybrid_metrics["precision@5"], reranked_metrics["precision@5"]],
    "MRR":         [baseline.retrieval_metrics.get("mrr", 0), bm25_metrics["mrr"],
                    hybrid_metrics["mrr"], reranked_metrics["mrr"]],
})

print(comparison.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(comparison))
width = 0.25
bars1 = ax.bar([i - width for i in x], comparison["Recall@5"], width, label="Recall@5", color="steelblue")
bars2 = ax.bar(x, comparison["Precision@5"], width, label="Precision@5", color="coral")
bars3 = ax.bar([i + width for i in x], comparison["MRR"], width, label="MRR", color="mediumseagreen")
ax.set_xticks(list(x))
ax.set_xticklabels(comparison["Strategy"], rotation=10)
ax.set_ylabel("Score")
ax.set_title("Retrieval Strategy Comparison")
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(EVAL_DIR / "retrieval_comparison.png", dpi=150)
plt.show()

In [ ]:
# Save advanced RAG results
results = EvalResults(
    experiment_name="advanced_rag",
    retriever_type="hybrid_reranked",
    embed_model=EMBED_MODEL,
    llm_model="granite4.1:8b",
    retrieval_metrics=reranked_metrics,
    notes="Hybrid alpha=0.7 + cross-encoder/ms-marco-MiniLM-L-6-v2 (retrieve-20, rerank-to-5)",
)
results.save(EVAL_DIR / "02_advanced_rag.json")
print(results.summary())

---

## Lessons Learned from Advanced RAG

### What improved
- **BM25 catches exact-term queries** that dense retrieval ranks poorly. For queries containing rare model names, algorithm names, or acronyms, BM25 consistently outperforms dense.
- **Hybrid beats both individually** — the alpha-weighted fusion reliably improves Recall@5 because each retriever's strengths compensate for the other's weaknesses.
- **Reranking improves precision** — the cross-encoder sorts the top-20 so the single most relevant chunk is at rank 1, improving MRR significantly.

### What still can't be fixed here

**1. No quality gate** — we still always pass top-k chunks to the LLM, even if they score poorly. For queries outside the corpus domain, the retrieved chunks are irrelevant noise.

**2. No fallback** — if nothing relevant is in the ArXiv corpus (e.g. a paper from this week), the system says "I don't know" with no attempt to search elsewhere.

**3. Static pipeline** — every query goes through the same steps (retrieve → rerank → generate) regardless of query type. A simple factual lookup doesn't need agentic reasoning; a complex multi-hop question does.

→ All three are solved in notebook 03 with LangGraph.